# Лабораторная работа №2

## Выполнил студент группы 6401-010302D Афанасьев Всеволод

In [1]:
import os

input_paths = [
    "/content/posts_sample.xml",
    "/content/programming-languages.csv"
]

for path in input_paths:
    status = "Found" if os.path.exists(path) else "Missing"
    print(f"[{status}] {path}")

[Found] /content/posts_sample.xml
[Found] /content/programming-languages.csv


### Подключение к спарку

In [2]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import col, count, row_number
from pyspark.sql.window import Window
import re

spark_session = SparkSession.builder \
    .appName("lab2") \
    .master("local[*]") \
    .getOrCreate()

spark_context = spark_session.sparkContext

print("Spark session started successfully.")
print(f"Running Spark version: {spark_session.version}")

Spark session started successfully.
Running Spark version: 4.0.2


### Работа с csv файлом с языками программирования

In [3]:
lang_ref_df = spark_session.read.csv("/content/programming-languages.csv", header=True)

valid_langs = set(
    record["name"].strip().lower()
    for record in lang_ref_df.collect()
    if record["name"] is not None
)

allowed_languages_bc = spark_context.broadcast(valid_langs)

print(f"Reference languages loaded: {len(valid_langs)} items.")
print("Broadcast variable created for distributed lookup.")

Reference languages loaded: 698 items.
Broadcast variable created for distributed lookup.


### Работа с xml файлом

In [4]:
date_pattern = re.compile(r'CreationDate="([^"]+)"')
tags_pattern = re.compile(r'Tags="([^"]*)"')
tag_extract = re.compile(r"<([^>]+)>")

xml_lines_rdd = spark_context.textFile("/content/posts_sample.xml")

def process_xml_line(line):
    output_rows = []

    d_match = date_pattern.search(line)
    if not d_match:
        return output_rows

    current_year = int(d_match.group(1)[:4])

    if not (2010 <= current_year <= 2020):
        return output_rows

    t_match = tags_pattern.search(line)
    if not t_match:
        return output_rows

    raw_tags = t_match.group(1).replace("&lt;", "<").replace("&gt;", ">")
    found_tags = tag_extract.findall(raw_tags)

    known_langs = allowed_languages_bc.value
    for tag in found_tags:
        if tag.lower() in known_langs:
            output_rows.append(Row(year=current_year, language=tag.lower()))

    return output_rows

year_lang_pairs_rdd = xml_lines_rdd.flatMap(process_xml_line)

total_pairs = year_lang_pairs_rdd.count()
print(f"XML processing completed.")
print(f"Total valid (year, language) pairs extracted: {total_pairs}")

XML processing completed.
Total valid (year, language) pairs extracted: 8054


In [5]:
pairs_df = spark_session.createDataFrame(year_lang_pairs_rdd)

aggregated_stats = pairs_df.groupBy("year", "language").agg(count("*").alias("usage_count"))

year_window = Window.partitionBy("year").orderBy(col("usage_count").desc())

ranking_result = aggregated_stats.withColumn("rank", row_number().over(year_window)).filter(col("rank") <= 10).orderBy("year", "rank")

print("Ranking calculation finished.")
print(f"Total rows in final report: {ranking_result.count()}")
ranking_result.show(20, truncate=False)

Ranking calculation finished.
Total rows in final report: 100
+----+-----------+-----------+----+
|year|language   |usage_count|rank|
+----+-----------+-----------+----+
|2010|java       |52         |1   |
|2010|php        |46         |2   |
|2010|javascript |44         |3   |
|2010|python     |26         |4   |
|2010|objective-c|23         |5   |
|2010|c          |20         |6   |
|2010|ruby       |12         |7   |
|2010|delphi     |8          |8   |
|2010|bash       |3          |9   |
|2010|r          |3          |10  |
|2011|php        |102        |1   |
|2011|java       |93         |2   |
|2011|javascript |83         |3   |
|2011|python     |37         |4   |
|2011|objective-c|34         |5   |
|2011|c          |24         |6   |
|2011|ruby       |20         |7   |
|2011|perl       |9          |8   |
|2011|delphi     |8          |9   |
|2011|bash       |7          |10  |
+----+-----------+-----------+----+
only showing top 20 rows


### Проверка parquet отчёта

In [6]:
output_path = "annual_lang_ranking.parquet"

ranking_result.write.mode("overwrite").parquet(output_path)
print(f"Data saved to disk: {output_path}")

verification_df = spark_session.read.parquet(output_path)

verification_df.show(120, truncate=False)

spark_session.stop()

Data saved to disk: annual_lang_ranking.parquet
+----+-----------+-----------+----+
|year|language   |usage_count|rank|
+----+-----------+-----------+----+
|2010|java       |52         |1   |
|2010|php        |46         |2   |
|2010|javascript |44         |3   |
|2010|python     |26         |4   |
|2010|objective-c|23         |5   |
|2010|c          |20         |6   |
|2010|ruby       |12         |7   |
|2010|delphi     |8          |8   |
|2010|bash       |3          |9   |
|2010|r          |3          |10  |
|2011|php        |102        |1   |
|2011|java       |93         |2   |
|2011|javascript |83         |3   |
|2011|python     |37         |4   |
|2011|objective-c|34         |5   |
|2011|c          |24         |6   |
|2011|ruby       |20         |7   |
|2011|perl       |9          |8   |
|2011|delphi     |8          |9   |
|2011|bash       |7          |10  |
|2012|php        |154        |1   |
|2012|javascript |132        |2   |
|2012|java       |124        |3   |
|2012|python    